![MSE Logo](https://moodle.msengineering.ch/pluginfile.php/1/core_admin/logo/0x150/1643104191/logo-mse.png)

# AdvNLP Lab (Graded Lab): Experimenting with Retrieval as Part of a RAG System

Total: 44 points

**Objectives:** We build the retrieval part of a RAG system and compare performance of classic KNN retrieval with additional cross encoder reranking. Eventually, we write two prompts for generation and test it on a LLM.

**Useful documentation:** Since you'll use LangChain for this assignment, [their documentation](https://python.langchain.com/docs/introduction/) might be helpful.

## Students

Name Student 1, Name Student 2

## Setup

First, we need to install the required packages for this assignment.

In [18]:
#%pip install pandas langchain-community langchain-classic langchain-huggingface faiss-cpu --quiet

In [19]:
import pandas as pd
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

We will use the [DRAGONBall Dataset](https://github.com/OpenBMB/RAGEval) as a basis for this assignment and load a subset of their documents. These will be the stored knowledge of the RAG system. To store them into the vector store, we will later directly create embeddings out of them, since they have alredy the size of suitable chunks. Each document consists of a unique ID and the actual content.

In [20]:
documents = pd.read_csv('docs.csv', index_col=0)
documents

,content
id,
40,Acme Government Solutions is a government indu...
41,Entertainment Enterprises Inc. is an entertain...
42,"Advanced Manufacturing Solutions Inc., establi..."
43,"EcoGuard Solutions, established on April 15, 2..."
44,"Green Fields Agriculture Ltd., established on ..."
...,...
211,Hospitalization Record:\n\nBasic Information:\...
212,**Hospitalization Record**\n\n**Basic Informat...
213,Hospitalization Record\n\nBasic Information:\n...


The main goal of the assignment is to evaluate the retrieval component of the RAG system. For that, we also load a dataset of queries, which we can use to retrieve matching documents. Each query has also assigned an array of documents in the form of their IDs, which match with the documents loaded before. We can use these to evaluate whether the correct documents were found by the retrieval or not.

In [21]:
queries = pd.read_csv('queries.csv', index_col=0)
queries['ground_truth_doc_ids'] = queries['ground_truth_doc_ids'].apply(lambda x: x.split(';'))
queries

,query,ground_truth_doc_ids
query_id,,
2286,When was Sparkling Clean Housekeeping Services...,[64]
2433,How did HealthPro Innovations' strategic partn...,[54]
6266,According to the hospitalization records of Br...,[212]
4499,"According to the judgment of Norwood, Unionvil...",[124]
2448,Based on HealthLife Solutions' 2020 corporate ...,[73]
...,...,...
2186,How did the severe drought in August 2018 lead...,[65]
3251,Compare the large-scale financing activities o...,"[58, 55]"
2268,How did CleanCo Housekeeping Services' investm...,[47]


## 1. Recall@N

**1a) [2 points]** We will evaluate the retrieval by comparing the retrieved documents with the ground truth documents assigned to the query. For that, we will use the Recall@N metric. Please describe in 1-2 sentences how we can interpret this metric in our case.

**Your Answer:** Recall@N in this context measures the percentage of relevant "ground truth" documents for a given query that the RAG system successfully fetches within its top N retrieved results.



**1b) [4 points]** Implement the Recall@N metric and test it with the following code.

In [22]:
def recall_at_n(retrieved_docs, relevant_doc_ids, n):
    """
    Calculate Recall@N.

    Parameters:
    - retrieved_docs: Sorted list of retrieved documents as LangChain Document objects
    - relevant_doc_ids: List of relevant document IDs
    - n: Number of top documents to consider

    Returns:
    - Recall@N
    """

    # TODO YOUR CODE HERE
    top_n_docs = retrieved_docs[:n]
    retrieved_ids = []
    
    for doc in top_n_docs:
        # Handles 'id' from the dummy test case or 'source' from CSVLoader 
        doc_id = doc.metadata.get('id') or doc.metadata.get('source')
        if doc_id is None:
            # Fallback for standard CSVLoader where content contains the ID
            for line in doc.page_content.split('\n'):
                if line.startswith('id: '):
                    doc_id = line.split(': ', 1)[1].strip()
                    break
        retrieved_ids.append(str(doc_id))
        
    relevant_set = set(str(rid) for rid in relevant_doc_ids)
    retrieved_set = set(retrieved_ids)
    
    hits = len(relevant_set.intersection(retrieved_set))
    return hits / len(relevant_set) if relevant_set else 0.0

In [23]:
### Test

recall_at_n(
    [Document(page_content='', metadata={'id': str(id)}) for id in range(10)],
    ['0', '1', '20'],
    3
)

0.6666666666666666

## 2. Embedding Model

**2a) [3 points]** Each document will be converted to an embedding representing the semantic meaning of the document. In this assignment, we will use model `sentence-transformers/all-MiniLM-L6-v2` from HuggingFace. Please answer the following questions about this model:

**Your Answers:**

Embedding Length: 384

Number of Parameters: 22.7M

Maximum Sequence Length: 256

## 3. Vector Store

**3a) [4 points]** Use LangChain to create a FAISS vector store and embed the documents with the above-mentioned embedding model. Load the documents again but this time with a Loader object from LangChain. Eventually, print the number of documents in the vector store.

In [24]:
# TODO YOUR CODE HERE
# Load documents using CSVLoader, mapping the 'id' column to the source metadata
loader = CSVLoader(file_path='docs.csv', source_column='id')
docs = loader.load()

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(docs, embeddings)

print(f"Number of documents in the vector store: {vector_store.index.ntotal}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7442.60it/s]


Number of documents in the vector store: 108


**3b) [3 points]** Retrieve the Top-3 documents for this query: "According to the hospitalization records of Bridgewater General Hospital, summarize the present illness of J. Reyes." and print the documents' ID and L2 distance.

In [25]:
# TODO YOUR CODE HERE
query_3b = "According to the hospitalization records of Bridgewater General Hospital, summarize the present illness of J. Reyes."
top_k = 3

# Perform similarity search with score (L2 distance for FAISS)
results_3b = vector_store.similarity_search_with_score(query_3b, k=top_k)

print(f"Query: {query_3b}\n")
for i, (doc, score) in enumerate(results_3b):
    doc_id = doc.metadata.get('source', 'Unknown ID')
    print(f"Rank {i+1}: Document ID: {doc_id} | L2 Distance: {score:.4f}")

Query: According to the hospitalization records of Bridgewater General Hospital, summarize the present illness of J. Reyes.

Rank 1: Document ID: 212 | L2 Distance: 0.7341
Rank 2: Document ID: 213 | L2 Distance: 1.0004
Rank 3: Document ID: 210 | L2 Distance: 1.0211


**3c) [2 points]** Check and show if a suitable document is found for the query in the Top-3 retrieved documents and show the relevant ones.

In [26]:
# TODO YOUR CODE HERE
# 1. Find the correct ground truth by matching keywords instead of the exact string
matched_query = queries[
    queries['query'].str.contains('Bridgewater General Hospital', case=False, na=False) & 
    queries['query'].str.contains('J. Reyes', case=False, na=False) &
    ~queries['query'].str.contains('L. Brown', case=False, na=False) # Exclude the query asking for 2 patients
]

# 2. Extract the ground truth IDs safely
ground_truth_3c = matched_query['ground_truth_doc_ids'].values[0]
print(f"Ground truth Document IDs: {ground_truth_3c}")

# 3. Extract the retrieved IDs from the results of 3b
retrieved_ids = [str(doc.metadata.get('source')) for doc, score in results_3b]
print(f"Retrieved Document IDs in Top-3: {retrieved_ids}")

# 4. Check if there is an intersection
hits = set(ground_truth_3c).intersection(set(retrieved_ids))
if hits:
    print(f"Yes, suitable documents were found in the Top-3. Relevant IDs: {list(hits)}")
else:
    print("No suitable document was found in the Top-3.")

Ground truth Document IDs: ['212']
Retrieved Document IDs in Top-3: ['212', '213', '210']
Yes, suitable documents were found in the Top-3. Relevant IDs: ['212']


**Your Answer:**
Yes, the suitable document (ID 212) was successfully retrieved in the Top-3. In fact, despite the query in 3b being phrased slightly differently than the ground-truth query in the dataset ("summarize the present illness" vs "what is the chief complaint"), the semantic embedding still accurately recognized the context and ranked document 212 at Rank 1.


## 4. Vector Store Evaluation

**4a) [4 points]** Now, we will search with each of the queries for the most relevant documents in the vector store, and calculate Recall@N with them and the assigned ground truth document IDs. To aggregate the results over all queries, we will calculate the mean. We will do this 3 times to and use a different value for $N$ each time: $N \in \{ 1, 3, 5, 25\}$.

In [27]:
# TODO YOUR CODE HERE
n_values = [1, 3, 5, 25]
recall_scores = {n: [] for n in n_values}

for index, row in queries.iterrows():
    query_text = row['query']
    ground_truth = row['ground_truth_doc_ids']
    
    # Retrieve top 25 docs (the maximum N we need to evaluate)
    retrieved_docs = vector_store.similarity_search(query_text, k=max(n_values))
    
    # Calculate Recall@N for each N
    for n in n_values:
        recall = recall_at_n(retrieved_docs, ground_truth, n)
        recall_scores[n].append(recall)

print("Vector Store Recall Evaluation:")
for n in n_values:
    mean_recall = sum(recall_scores[n]) / len(recall_scores[n])
    print(f"Mean Recall@{n}: {mean_recall:.4f}")

Vector Store Recall Evaluation:
Mean Recall@1: 0.6800
Mean Recall@3: 0.8100
Mean Recall@5: 0.8650
Mean Recall@25: 0.9850


**4b) [2 points]** When looking at the four calculated Recall@N scores, what do you observe and how can you explain this?

**Your Answer:**
As $N$ increases, the Mean Recall@N score strictly increases (from 0.6800 at Recall@1 to 0.9850 at Recall@25). This is expected because retrieving a larger pool of documents gives the system more chances to capture the relevant ground-truth document. The gap between Recall@1 and Recall@25 indicates that while the base embedding model is excellent at finding the correct document within the broader top 25, it frequently struggles to rank it exactly at position 1 based solely on vector distance.


## 5. Cross Encoder

**5a) [3 points]** We want to use a cross encoder model to rerank the retrieved documents. Describe in 1-2 sentences how a new document order can be determined using a cross encoder.

**Your Answer:** A cross-encoder determines a new document order by passing both the query and the document simultaneously through its transformer network (e.g., separated by a [SEP] token). This allows the model to compute rich, token-level attention interactions between the query and the document text, yielding a highly accurate relevance score compared to comparing separated vectors.



**5b) [4 points]** Now again, we want to calculate Recall@N for all queries and the same $N$ as before. This time, we want to rerank the Top-25 retrieved documents using the cross encoder model `BAAI/bge-reranker-base`. Implement this using LangChain components and report the average Recall for $N \in \{ 1, 3, 5, 25\}$.

In [28]:
# TODO YOUR CODE HERE
# Initialize the cross-encoder and the LangChain reranker compressor
cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
compressor = CrossEncoderReranker(model=cross_encoder, top_n=max(n_values))

rerank_recall_scores = {n: [] for n in n_values}

for index, row in queries.iterrows():
    query_text = row['query']
    ground_truth = row['ground_truth_doc_ids']
    
    # 1. Base retrieval: get top 25 from vector store
    base_retrieved_docs = vector_store.similarity_search(query_text, k=max(n_values))
    
    # 2. Rerank the top 25 documents using the cross-encoder
    reranked_docs = compressor.compress_documents(base_retrieved_docs, query_text)
    
    # 3. Calculate Recall@N on the reranked list
    for n in n_values:
        recall = recall_at_n(reranked_docs, ground_truth, n)
        rerank_recall_scores[n].append(recall)

print("Cross-Encoder Reranked Recall Evaluation:")
for n in n_values:
    mean_recall = sum(rerank_recall_scores[n]) / len(rerank_recall_scores[n])
    print(f"Mean Reranked Recall@{n}: {mean_recall:.4f}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 19877.28it/s]


Cross-Encoder Reranked Recall Evaluation:
Mean Reranked Recall@1: 0.7450
Mean Reranked Recall@3: 0.9550
Mean Reranked Recall@5: 0.9750
Mean Reranked Recall@25: 0.9850


**5c) [2 points]** What do you observe when you compare the Recall@N scores after reranking with the scores without reranking? Write 1-2 sentences about this and why this might happen.

**Your Answer:**
After reranking with the cross-encoder, the Recall scores for smaller $N$ improved significantly (e.g., Recall@1 jumped from 0.6800 to 0.7450, and Recall@3 from 0.8100 to 0.9550). This shows the cross-encoder is much better at capturing semantic nuances to push truly relevant documents to the very top. Noticeably, Recall@25 remained identical at 0.9850; this happens because the reranker only re-sorts the initial 25 documents fetched by the vector store, it cannot retrieve new documents that weren't already in that initial pool.

## 6. Generation

**6a) [6 points]** After improving the retrieval part of the RAG system, we want to finally generate an answer for our query. Retrieve the most relevant document for query "How much funding did HealthPro Innovations raise in February 2021?" and print its ID. Then write the instruction message of a prompt to answer this query including all necessary elements before running it using your favourite LLM (ChatGPT GPT-4o, etc.). Please paste the answer from the model and indicate which model you used.

In [ ]:
# TODO YOUR CODE HERE


**Your Prompt:**



**Generated Answer:**



**Used Model:**



**6b) [3 points]** We want to use in-context learning and provide the LLM one example of a possible answer. Use the same prompt and extend it, that it should follow this example answer: "Yep, they sold a lot in that year. Over 50 million units as I can see — pretty big move, respect!". Use the same model, create a fresh chat and run this new prompt. Highlight the changes in the prompt using **bold style** or <span style="color:red;">color</span>.

**Your Prompt:**



**Generated Answer:**



**6c) [2 points]** Please check if the two answers are correct according to the document and how they differ. Does the model follow the example in the second prompt?

**Your Answer:**


## End of AdvNLP Lab

Please make sure all cells have been executed, save this completed notebook, and upload it to Moodle.